# Training-only detrending triage for legacy EXP1 inputs

## Executive finding

The previous input2, input3a-input3d, and historical input4 fields **do contain train/validation/test leakage** because their monthly quadratic trends were fitted to all 14 selected ensemble members. None, however, shows the catastrophic numerical contamination found for icethick.

Refitting on only the eight training members changes the legacy fields by **3.4-4.1% of their RMS magnitude**, with maximum pointwise changes of **0.027-0.056**. These are scientifically meaningful systematic shifts, but not numerical degeneration.

The SST concern is partly confirmed: validation SST has a rare tail up to about **8.2**, but it exists under both detrending fits and does not contaminate training after the correction. Training SST has no values above 1. This notebook is read-only and uses normalized triage fields; it creates no model-ready pairs.

In [ ]:
from pathlib import Path
import hashlib
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

repo_root = next(q for q in [Path.cwd(), *Path.cwd().parents] if (q / "src").is_dir())
sys.path.insert(0, str(repo_root))
from src import config_cesm
from src.experiment_configs import load_config

root = Path(config_cesm.PROCESSED_DATA_DIRECTORY) / "normalization_triage" / "exp1_legacy_train_only_detrend"
variants = ("input2", "input3a", "input3b", "input3c", "input3d", "input4")
variables = ("icefrac", "sst", "psl", "z500", "t2m")
suffixes = ("norm_train_only_detrend.nc", "detrend_coeffs_train_only.nc",
            "comparison_summary.csv", "comparison_maps.nc", "metadata.json")
required = [root / f"{v}_{s}" for v in variables for s in suffixes]
assert all(q.exists() for q in required), [str(q) for q in required if not q.exists()]
print(root)
print(f"Validated {len(required)} generated artifacts.")

## Design

Legacy scaling was already training-only: icefrac subtracts a training calendar-month/grid-cell mean; the other fields use training calendar-month/grid-cell min-max scaling. This experiment repeats that scaling and changes only detrending:

- Legacy: fit each calendar-month/grid-cell quadratic to the mean of all 14 members.
- Corrected: fit the same quadratic to the eight training members, then apply it unchanged to all splits.

All configurations share the split and variable recipes, so one corrected file is written per unique variable. The generator then compares it with every configuration-specific legacy file. Synthetic tests verify that the isolated implementation exactly matches the legacy function when all members are supplied.

In [ ]:
coverage_rows, legacy_paths = [], {}
for variant in variants:
    cfg = load_config(f"exp1_inputs:{variant}")
    for variable, settings in cfg.input_config.items():
        if settings["include"] and settings["norm"] and not settings["auxiliary"]:
            coverage_rows.append({
                "configuration": variant, "variable": variable,
                "legacy folder": cfg.data_name,
                "scaling": "min-max" if settings["use_min_max"] else "mean removal",
            })
            legacy_paths[(variant, variable)] = (
                Path(config_cesm.PROCESSED_DATA_DIRECTORY) / "normalized_inputs"
                / cfg.data_name / f"{variable}_norm.nc"
            )
coverage = pd.DataFrame(coverage_rows)
display(coverage)
print("Configuration-variable comparisons:", len(coverage))

### Duplicate-file verification

The legacy pipeline wrote identical variables into configuration-specific folders. This hash check proves those files are byte-identical, justifying one corrected counterfactual per variable. It reads about 10 GB but uses little memory.

In [ ]:
hash_rows = []
for (variant, variable), path in legacy_paths.items():
    with path.open("rb") as f:
        digest = hashlib.file_digest(f, "sha256").hexdigest()
    hash_rows.append({"configuration": variant, "variable": variable, "sha256": digest})
hashes = pd.DataFrame(hash_rows)
display(hashes)
assert (hashes.groupby("variable").sha256.nunique() == 1).all()

## 1. Leakage magnitude

The difference between fitted trends is broadcast to every member, so its RMSE is nearly identical across splits. Training rows give the clearest scale comparison. A 3-4% perturbation is not blowup, but it is not negligible numerical noise.

In [ ]:
summary = pd.concat(
    [pd.read_csv(root / f"{v}_comparison_summary.csv") for v in variables],
    ignore_index=True,
)
assert (summary.finite_status_mismatches == 0).all()
metrics = [c for c in summary if c not in {"configuration", "variable", "split"}]
assert (summary.groupby(["variable", "split"])[metrics].nunique() <= 1).all().all()
canonical = summary.drop_duplicates(["variable", "split"])
train = canonical.query("split == 'train'").copy()
train["difference RMS / old RMS (%)"] = 100 * train.difference_rmse_fraction_of_old_rms
overview = train[["variable", "old_rms", "new_rms", "difference_rmse",
                  "difference RMS / old RMS (%)", "difference_max_abs",
                  "old_max_abs", "new_max_abs"]].set_index("variable")
display(overview.style.format("{:.6g}"))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
overview["difference RMS / old RMS (%)"].plot.bar(ax=ax[0], color="#4472C4")
overview.difference_max_abs.plot.bar(ax=ax[1], color="#ED7D31")
ax[0].set(title="Fit change relative to legacy RMS", ylabel="percent")
ax[1].set(title="Maximum pointwise change", ylabel="normalized units")
for a in ax:
    a.tick_params(axis="x", rotation=0)
    a.grid(axis="y", alpha=.25)
fig.tight_layout();

SST changes most in relative terms (4.11% of legacy RMS), followed by icefrac (3.71%), t2m (3.65%), psl (3.44%), and z500 (3.40%). The largest individual difference is 0.0558 for icefrac.

All prior configurations are affected. Importantly, icefrac_norm.nc supplies both lagged SIC inputs and the anomaly-target source. A leakage-free rerun would therefore require regenerating both inputs and targets/pairs. This triage intentionally does neither.

In [ ]:
by_split = canonical[["variable", "split", "old_max_abs", "new_max_abs",
                      "difference_rmse", "difference_rmse_fraction_of_old_rms",
                      "difference_max_abs", "finite_status_mismatches"]].copy()
by_split["difference RMS / old RMS (%)"] = 100 * by_split.pop("difference_rmse_fraction_of_old_rms")
display(by_split.set_index(["variable", "split"]).style.format("{:.6g}"))

## 2. SST: rare held-out extrapolation, not blowup feedback

Training SST remains below 1 under both fits. Validation/test values can exceed 1 because held-out physical SST lies outside a narrow training min-max interval near the ice edge. Training-only detrending cannot remove an extrapolation created by the preceding scaling step.

In [ ]:
sst = canonical.query("variable == 'sst'").copy()
display(sst[["split", "old_finite_values", "old_max_abs", "new_max_abs",
             "old_abs_gt_1", "new_abs_gt_1", "old_abs_gt_2", "new_abs_gt_2",
             "old_abs_gt_5", "new_abs_gt_5", "difference_max_abs"]].set_index("split"))
fractions = pd.DataFrame({
    "old fraction |SST| > 1": sst.set_index("split").old_abs_gt_1 / sst.set_index("split").old_finite_values,
    "new fraction |SST| > 1": sst.set_index("split").new_abs_gt_1 / sst.set_index("split").new_finite_values,
})
display(fractions.style.format("{:.3e}"))

In [ ]:
cfg = load_config("exp1_inputs:input3a")
with xr.open_dataset(root / "sst_comparison_maps.nc") as ds:
    m = ds.new_max_abs.sel(configuration="input3a", split="val").values
yi, xi = np.unravel_index(np.nanargmax(m), m.shape)
legacy_dir = Path(config_cesm.PROCESSED_DATA_DIRECTORY) / "normalized_inputs" / cfg.data_name
with xr.open_dataset(legacy_dir / "sst_norm.nc") as old_ds, \
     xr.open_dataset(root / "sst_norm_train_only_detrend.nc") as new_ds:
    old = old_ds.sst.sel(member_id=cfg.data_split["val"]).isel(y=yi, x=xi).values
    new = new_ds.sst.sel(member_id=cfg.data_split["val"]).isel(y=yi, x=xi).values
    mi, ti = np.unravel_index(np.nanargmax(np.abs(new)), new.shape)
    member, time = cfg.data_split["val"][mi], pd.Timestamp(new_ds.time.values[ti])
    old_value, new_value = float(old[mi, ti]), float(new[mi, ti])
raw_path = Path(config_cesm.DATA_DIRECTORY) / "cesm_data" / "sst" / "sst_combined.nc"
with xr.open_dataset(legacy_dir / "sst_min.nc") as mn_ds, \
     xr.open_dataset(legacy_dir / "sst_max.nc") as mx_ds, \
     xr.open_dataset(raw_path) as raw_ds:
    mn = mn_ds.sst.sel(month=time.month).isel(y=yi, x=xi).item()
    mx = mx_ds.sst.sel(month=time.month).isel(y=yi, x=xi).item()
    raw = raw_ds.sst.sel(member_id=member, time=time).isel(y=yi, x=xi).item()
scaled = (raw - mn) / (mx - mn)
display(pd.Series({
    "member": member, "time": time, "y index": yi, "x index": xi,
    "raw SST": raw, "training monthly minimum": mn, "training monthly maximum": mx,
    "training range": mx - mn, "scaled before detrending": scaled,
    "legacy all-member detrended": old_value, "training-only detrended": new_value,
    "fit-partition difference": new_value - old_value,
}).to_frame("value"))

At the worst validation point, raw SST is -1.558 while the training-member June interval is only -1.893 to -1.854 (range 0.0386). Scaling gives 8.657. Legacy and corrected detrending give 8.186 and 8.203: the 0.017 difference is leakage, while the value near 8 is min-max extrapolation. This differs qualitatively from icethick, where a denominator near \(10^{-16}\) produced values near \(10^{13}\) and contaminated training near \(10^{10}\).

## 3. Spatial pattern of training changes

Each map is the maximum absolute legacy-versus-corrected difference over all training members and months. It describes the fitted-trend change, not isolated samples.

In [ ]:
representative = {"icefrac": "input2", "sst": "input3a", "psl": "input3b",
                  "z500": "input3c", "t2m": "input3d"}
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for axis, variable in zip(axes.flat, variables):
    with xr.open_dataset(root / f"{variable}_comparison_maps.nc") as ds:
        field = ds.max_abs_difference.sel(
            configuration=representative[variable], split="train"
        ).values
    image = axis.imshow(field, origin="lower", cmap="magma")
    axis.set_title(f"{variable}: max |difference|")
    fig.colorbar(image, ax=axis, shrink=.8)
axes.flat[-1].axis("off")
fig.tight_layout();

## Conclusions

1. **Leakage is present in every prior configuration.** Corrected fields differ by 3.4-4.1% of legacy RMS. Through icefrac, leakage affects both predictors and anomaly targets.
2. **No prior variable reproduces the icethick numerical failure.** Training maxima stay below 1, finite masks are unchanged, and leakage-induced shifts stay below 0.056.
3. **SST has a rare held-out tail.** Validation has 14 values above 5 among about 20 million finite values; training has none above 1. This is scaling extrapolation, not blowup feedback into training.
4. **Training-only detrending is necessary but not sufficient to bound held-out min-max values.** Tiny training ranges and out-of-range policies remain separate scientific choices.
5. **Model impact is not measured here.** Leakage-free comparisons would require regenerated pairs and retraining.

In [ ]:
artifacts = pd.DataFrame([
    {"artifact": q.name, "size (MB)": q.stat().st_size / 1e6}
    for q in sorted(root.iterdir()) if q.is_file()
])
display(artifacts.style.format({"size (MB)": "{:.3f}"}))
print(f"Total generated size: {artifacts['size (MB)'].sum() / 1000:.3f} GB")
print("Generation job 42550708; enriched-summary job 42551790")